In [95]:
from sklearn.mixture import GaussianMixture
from app.preprocessing.preprocessing import preprocessing , PCA_transform
from models.gmm_model import train_gmm , predict_gmm, GMM_CONFIGS
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import pairwise_distances

In [63]:
df = pd.read_csv("../cross_genre_music_discovery/raw_data/spotify-tracks-dataset-detailed.csv")

X_scaled, df_metadata, scaler, features = preprocessing(df)

In [64]:
df_scaled = pd.DataFrame(X_scaled, columns=features)
#df_completo = pd.concat([df_metadata, X_scaled], axis=1)
#df_completo.head()

In [65]:
df_completo = pd.concat([df_metadata, df_scaled], axis=1)
df_completo.head()

,track_id,artists,track_name,genres,danceability,energy,loudness,speechiness,acousticness,instrumentalness,valence,tempo
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,"[acoustic, j-pop, singer-songwriter, songwriter]",0.642976,-0.671544,0.331501,0.761537,-0.866112,-0.539305,0.931651,-1.143959
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost - Acoustic,"[acoustic, chill]",-0.800829,-1.818558,-1.676941,-0.043998,1.782308,-0.539291,-0.769508,-1.490966
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,[acoustic],-0.699312,-1.068138,-0.240643,-0.292784,-0.338091,-0.539308,-1.327701,-1.529466
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Can't Help Falling In Love,[acoustic],-1.669368,-2.232261,-1.922037,-0.527078,1.725883,-0.539091,-1.240365,1.978131
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,[acoustic],0.315863,-0.741531,-0.230495,-0.330223,0.431074,-0.539308,-1.149232,-0.078050


In [66]:
df_completo.shape

(88751, 12)

In [67]:
df_pca3 = PCA_transform(X_scaled)

AttributeError: 'numpy.ndarray' object has no attribute 'index'

In [68]:
df_metadata

,track_id,artists,track_name,genres
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,"[acoustic, j-pop, singer-songwriter, songwriter]"
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost - Acoustic,"[acoustic, chill]"
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,[acoustic]
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Can't Help Falling In Love,[acoustic]
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,[acoustic]
...,...,...,...,...
88746,2C3TZjDRiAzdyViavDJ217,Rainy Lullaby,Sleep My Little Boy,[world-music]
88747,1hIz5L4IB9hN3WRYPOCGPw,Rainy Lullaby,Water Into Light,[world-music]
88748,6x8ZfSoqDjuNa5SVP5QjvX,Cesária Evora,Miss Perfumado,[world-music]
88749,2e6sXL2bYv4bSz6VTdnfLs,Michael W. Smith,Friends,[world-music]


In [69]:
["Comedy","Ghost - Acoustic", "To Begin Again", "Can't Help Falling In Love", "Hold On","Sleep My Little Boy","Water Into Light","Miss Perfumado","Friends","Barbincor"]

['Comedy',
 'Ghost - Acoustic',
 'To Begin Again',
 "Can't Help Falling In Love",
 'Hold On',
 'Sleep My Little Boy',
 'Water Into Light',
 'Miss Perfumado',
 'Friends',
 'Barbincor']

# Criando função que recebe a playlist

In [70]:
def input_tracks(tracks):
    if type(tracks) != list:
        raise TypeError("O input deve ser uma lista")
    if len(tracks) != 10:
        raise ValueError("A lista deve conter 10 músicas")

    for music in tracks:

        if (
            df_completo["track_id"].eq(music).any()
            or df_completo["track_name"].eq(music).any()
        ):
            pass
        else:
            print(f"Música não encontrada: {music}")

    return tracks

# Testando possíveis erros

In [71]:
tracks = ["Hold On"]

In [72]:
input_tracks(tracks)

ValueError: A lista deve conter 10 músicas

In [73]:
tracks_2 = 2

In [74]:
input_tracks(tracks_2)

TypeError: O input deve ser uma lista

In [75]:
tracks_3 = ["Comedy","Ghost - Acoustic", "To Begin Again", "Can't Help Falling In Love", "Hold On","Sleep My Little Boy","Water Into Light","Miss Perfumado","Friends","Barbincor"]

In [76]:
type(tracks_3)

list

In [77]:
input_tracks(tracks_3)

['Comedy',
 'Ghost - Acoustic',
 'To Begin Again',
 "Can't Help Falling In Love",
 'Hold On',
 'Sleep My Little Boy',
 'Water Into Light',
 'Miss Perfumado',
 'Friends',
 'Barbincor']

In [78]:
df_completo["cluster_gmm"] = model.predict(df_pca3)

NameError: name 'model' is not defined

In [79]:
type(X_scaled)

numpy.ndarray

# Verificando os clusters

In [80]:
modelos = {}

for nome, cfg in GMM_CONFIGS.items():

    n_pca = 3 if nome.startswith("pca3") else 5

    # Transformar os dados e guardar o PCA ajustado
    df_pca, pca_ajustado = PCA_transform(
        df_scaled,
        number_of_components=n_pca
    )

    modelos[nome] = {
        "model": train_gmm(df_pca, **cfg),
        "pca": pca_ajustado,
        "scaler": scaler
    }

In [81]:
modelos

{'pca3_tied15': {'model': GaussianMixture(covariance_type='tied', n_components=15, n_init=5,
                  random_state=42),
  'pca': PCA(n_components=3),
  'scaler': StandardScaler()},
 'pca5_diag30': {'model': GaussianMixture(covariance_type='diag', n_components=30, n_init=5,
                  random_state=42),
  'pca': PCA(n_components=5),
  'scaler': StandardScaler()}}

# Verificação de clusters

In [82]:
tracks_teste = (
    df_completo["track_id"]
    .dropna()
    .drop_duplicates()
    .sample(10, random_state=42)
    .tolist()
)

print(tracks_teste)
print("Quantidade:", len(tracks_teste))

['5MmDAFQa8SP9bA5J3f5vYc', '4dqqpOUXW63UYb0zEbc1QL', '7fag34uWWmEBYo6SAhFIT9', '6weT0OACEb3e4KLQJInf4I', '4zwthrpGFN6edk5u7FptKJ', '2af6mkiXvN3sKidbc2EAGr', '1oUmQdyZJGKi8EZKvNrRZQ', '0BFtrq3fyv1kRcId4o18mK', '3KLrPoysdfLM82TPyZHiBi', '7voJiEkIHuBZ7EKJC4EabO']
Quantidade: 10


In [83]:
input_df = df_completo[
    df_completo["track_id"].isin(tracks_teste)
].copy()

print("Formato:", input_df.shape)
print("IDs únicos:", input_df["track_id"].nunique())

Formato: (10, 12)
IDs únicos: 10


In [84]:
input_df[
    ["track_id", "track_name", "artists", "genres"]
]

,track_id,track_name,artists,genres
8563,0BFtrq3fyv1kRcId4o18mK,Um Drink no Inferno,Xamã;Bagua Records;Neo Beats,"[brazil, funk]"
14117,3KLrPoysdfLM82TPyZHiBi,Where Is Thumbkin,The Wiggles,[children]
18726,4zwthrpGFN6edk5u7FptKJ,Think of Christmas,Anne-Marie,"[dance, house]"
23253,6weT0OACEb3e4KLQJInf4I,Olivia,Die Zipfelbuben;DJ Cashi,"[disco, party]"
28296,5MmDAFQa8SP9bA5J3f5vYc,Origin,ARTBAT;Shall Ocin;braev,[electronic]
41314,7voJiEkIHuBZ7EKJC4EabO,Vícios E Virtudes - Ao Vivo,Charlie Brown Jr.,[hard-rock]
49745,4dqqpOUXW63UYb0zEbc1QL,The Plague 19,Winterise,[iranian]
54752,1oUmQdyZJGKi8EZKvNrRZQ,Chuttesai Chuttesai,Yuvan Shankar Raja;Karthik;Sunitha Sarathy,[k-pop]
54900,2af6mkiXvN3sKidbc2EAGr,Aap Mujhe Achche Lagne Lage - Aap Mujhe Achche...,Alka Yagnik;Abhijeet,[k-pop]
82835,7fag34uWWmEBYo6SAhFIT9,John & Jill,Kikki Danielsson,[swedish]


In [85]:
print(modelos.keys())

dict_keys(['pca3_tied15', 'pca5_diag30'])


In [86]:
config = modelos["pca3_tied15"]

In [87]:
X_input = input_df[features]

In [88]:
X_input_scaled = config["scaler"].transform(X_input)

X_input_pca = config["pca"].transform(X_input_scaled)

/home/victor/.pyenv/versions/3.10.6/envs/cross_genre_music_discovery/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(


In [89]:
clusters = config["model"].predict(X_input_pca)

print(clusters)

[14 10  0  0  4  7 13  0 11  0]


/home/victor/.pyenv/versions/3.10.6/envs/cross_genre_music_discovery/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but GaussianMixture was fitted with feature names
  warnings.warn(


In [90]:
input_df["cluster_gmm"] = clusters

input_df[
    ["track_name", "artists", "genres", "cluster_gmm"]
]

,track_name,artists,genres,cluster_gmm
8563,Um Drink no Inferno,Xamã;Bagua Records;Neo Beats,"[brazil, funk]",14
14117,Where Is Thumbkin,The Wiggles,[children],10
18726,Think of Christmas,Anne-Marie,"[dance, house]",0
23253,Olivia,Die Zipfelbuben;DJ Cashi,"[disco, party]",0
28296,Origin,ARTBAT;Shall Ocin;braev,[electronic],4
41314,Vícios E Virtudes - Ao Vivo,Charlie Brown Jr.,[hard-rock],7
49745,The Plague 19,Winterise,[iranian],13
54752,Chuttesai Chuttesai,Yuvan Shankar Raja;Karthik;Sunitha Sarathy,[k-pop],0
54900,Aap Mujhe Achche Lagne Lage - Aap Mujhe Achche...,Alka Yagnik;Abhijeet,[k-pop],11
82835,John & Jill,Kikki Danielsson,[swedish],0


In [91]:
print("Músicas no input:", len(input_df))
print("Features utilizadas:", X_input.shape[1])
print("Componentes do PCA:", config["pca"].n_components_)
print("Gaussianas do GMM:", config["model"].n_components)

Músicas no input: 10
Features utilizadas: 8
Componentes do PCA: 3
Gaussianas do GMM: 15


In [99]:
grupos = input_df.groupby("cluster_gmm")

for cluster, grupo in grupos:
    print(
        f"Cluster {cluster}: "
        f"{len(grupo)} música(s)"
    )

Cluster 0: 4 música(s)
Cluster 4: 1 música(s)
Cluster 7: 1 música(s)
Cluster 10: 1 música(s)
Cluster 11: 1 música(s)
Cluster 13: 1 música(s)
Cluster 14: 1 música(s)


In [100]:
grupo_0 = input_df[
    input_df["cluster_gmm"] == 0
]

In [101]:
df_coordenadas = pd.DataFrame(
    X_input_pca,
    index=input_df.index
)

In [102]:
coordenadas_0 = df_coordenadas.loc[grupo_0.index]

In [103]:
distancias_0 = pairwise_distances(
    coordenadas_0,
    metric="euclidean"
)

In [104]:
soma_distancias = distancias_0.sum(axis=1)

posicao_medoide = soma_distancias.argmin()

medoide = grupo_0.iloc[posicao_medoide]

In [105]:
print("Medoide:", medoide["track_name"])
print("Artista:", medoide["artists"])
print("Cluster:", medoide["cluster_gmm"])

Medoide: Chuttesai Chuttesai
Artista: Yuvan Shankar Raja;Karthik;Sunitha Sarathy
Cluster: 0


In [ ]:
def find_medoids(input_df, X_input, cluster_col="cluster_gmm"):

    # Criar um DataFrame com as coordenadas das músicas
    df_coordenadas = pd.DataFrame(
        X_input,
        index=input_df.index
    )

    medoides = []

    # Percorrer os clusters presentes no input
    for cluster, grupo in input_df.groupby(cluster_col):

        # Selecionar as coordenadas das músicas do grupo
        coordenadas_grupo = df_coordenadas.loc[grupo.index]

        # Calcular as distâncias entre as músicas do grupo
        distancias = pairwise_distances(
            coordenadas_grupo,
            metric="euclidean"
        )

        # Somar as distâncias de cada música às demais
        soma_distancias = distancias.sum(axis=1)

        # Encontrar a posição da música mais central
        posicao_medoide = soma_distancias.argmin()

        # Recuperar o registro da música escolhida
        medoide = grupo.iloc[posicao_medoide].copy()

        # Guardar também o tamanho do grupo
        medoide["n_musicas"] = len(grupo)

        medoides.append(medoide)

    return pd.DataFrame(medoides).reset_index(drop=True)